# TensorRT spike on Kaggle (THROWAWAY)

Question: on a Kaggle T4, how much faster is TensorRT fp16 than ncnn for the same network, and is the picture the same?

The network is `realesr-animevideov3` (SRVGGNetCompact, official x4 weights), the same one `video2x` runs through ncnn.
Input: one 1080p frame cut from your clip. Output: 4320p (x4). Reference numbers already measured on a T4 (docs/hypotheses.md, T10): ncnn 1.13 fps, PyTorch fp32 0.56, fp16 2.53, fp16 channels_last 3.22.

Setup: Settings > Accelerator > GPU T4 x2 (one GPU is used), Internet on. Add your 10 s clip (`sample_10s.mp4`) as a Dataset, or the cell falls back to a synthetic frame (speed is valid, PSNR less meaningful).

In [ ]:
# 1. GPU check and install
import subprocess, sys, os
os.makedirs("/kaggle/temp", exist_ok=True)

def sh(c):
    r = subprocess.run(c, shell=True, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    print(out[-3000:])
    return r

sh("nvidia-smi --query-gpu=name,memory.total,driver_version,clocks.max.sm,power.limit --format=csv")
sh(f"{sys.executable} -m pip install -q tensorrt-cu12 onnx 2>&1 | tail -3")
import tensorrt as trt, torch
print("TensorRT", trt.__version__, "| torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))

In [ ]:
# 2. Input frame: 1080p
import glob, numpy as np
from PIL import Image

vids = sorted(glob.glob("/kaggle/input/**/*.mp4", recursive=True) + glob.glob("/kaggle/input/**/*.mkv", recursive=True))
frame_png = "/kaggle/temp/frame.png"
if vids:
    sh(f'ffmpeg -v error -y -ss 5 -i "{vids[0]}" -frames:v 1 -vf scale=1920:1080 {frame_png}')
    img = np.array(Image.open(frame_png).convert("RGB"))
    print("frame from", vids[0], img.shape)
else:
    rng = np.random.default_rng(0)
    img = (rng.random((1080, 1920, 3)) * 255).astype(np.uint8)
    print("no video found: synthetic frame (PSNR not meaningful)")
x32 = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).float().div(255).cuda()  # 1x3x1080x1920
print(x32.shape)

In [ ]:
# 3. Model: SRVGGNetCompact + official realesr-animevideov3 x4 weights, export to ONNX
import torch.nn as nn, torch.nn.functional as F

class SRVGGNetCompact(nn.Module):
    def __init__(self, num_in_ch=3, num_out_ch=3, num_feat=64, num_conv=16, upscale=4):
        super().__init__()
        self.upscale = upscale
        self.body = nn.ModuleList()
        self.body.append(nn.Conv2d(num_in_ch, num_feat, 3, 1, 1))
        self.body.append(nn.PReLU(num_parameters=num_feat))
        for _ in range(num_conv):
            self.body.append(nn.Conv2d(num_feat, num_feat, 3, 1, 1))
            self.body.append(nn.PReLU(num_parameters=num_feat))
        self.body.append(nn.Conv2d(num_feat, num_out_ch * upscale * upscale, 3, 1, 1))
        self.upsampler = nn.PixelShuffle(upscale)
    def forward(self, x):
        out = x
        for layer in self.body:
            out = layer(out)
        out = self.upsampler(out)
        return out + F.interpolate(x, scale_factor=self.upscale, mode="nearest")

pth = "/kaggle/temp/realesr-animevideov3.pth"
if not os.path.exists(pth):
    sh(f"wget -q -O {pth} https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-animevideov3.pth")
state = torch.load(pth, map_location="cpu")
state = state.get("params", state)
model = SRVGGNetCompact().eval()
model.load_state_dict(state)
model = model.cuda()

onnx_path = "/kaggle/temp/animevideov3_x4_1080p.onnx"
with torch.no_grad():
    ref32 = model(x32)                      # fp32 reference output
    torch.onnx.export(model, x32, onnx_path, opset_version=17, input_names=["input"], output_names=["output"])
print("ref", ref32.shape, "onnx", os.path.getsize(onnx_path) // 1024, "KiB")

In [ ]:
# 4. TensorRT engine (fp16) and timing helpers
import time

def build_engine(onnx_file, fp16=True, workspace_gb=4):
    logger = trt.Logger(trt.Logger.WARNING)
    builder = trt.Builder(logger)
    network = builder.create_network(0)
    parser = trt.OnnxParser(network, logger)
    if not parser.parse(open(onnx_file, "rb").read()):
        raise RuntimeError("\n".join(str(parser.get_error(i)) for i in range(parser.num_errors)))
    cfg = builder.create_builder_config()
    cfg.set_memory_pool_limit(trt.MemoryPoolType.WORKSPACE, workspace_gb << 30)
    if fp16:
        cfg.set_flag(trt.BuilderFlag.FP16)
    t = time.time()
    blob = builder.build_serialized_network(network, cfg)
    print(f"engine built in {time.time() - t:.0f} s, {len(blob) // 2**20} MiB")
    return trt.Runtime(logger).deserialize_cuda_engine(blob)

def run_engine(engine, x, iters=20, warmup=3):
    ctx = engine.create_execution_context()
    in_name, out_name = engine.get_tensor_name(0), engine.get_tensor_name(1)
    ctx.set_input_shape(in_name, tuple(x.shape))
    out = torch.empty(tuple(ctx.get_tensor_shape(out_name)), dtype=torch.float32, device="cuda")
    x = x.contiguous()
    ctx.set_tensor_address(in_name, x.data_ptr())
    ctx.set_tensor_address(out_name, out.data_ptr())
    stream = torch.cuda.Stream()
    with torch.cuda.stream(stream):
        for _ in range(warmup):
            ctx.execute_async_v3(stream.cuda_stream)
        stream.synchronize()
        t0, t1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        t0.record(stream)
        for _ in range(iters):
            ctx.execute_async_v3(stream.cuda_stream)
        t1.record(stream)
        stream.synchronize()
    return iters / (t0.elapsed_time(t1) / 1000), out

def psnr(a, b):
    mse = torch.mean((a.clamp(0, 1).double() - b.clamp(0, 1).double()) ** 2).item()
    return float("inf") if mse == 0 else 10 * np.log10(1 / mse)

results = {}
eng = build_engine(onnx_path)
fps, trt_out = run_engine(eng, x32)
results["TensorRT fp16 (fp32 I/O)"] = (fps, psnr(trt_out, ref32))
print(results)

In [ ]:
# 5. In-session control: PyTorch fp16 channels_last (should be near the 3.22 fps measured in T10)
def time_torch(m, x, iters=20, warmup=3):
    with torch.no_grad():
        for _ in range(warmup): m(x)
        torch.cuda.synchronize()
        t0, t1 = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
        t0.record()
        for _ in range(iters): out = m(x)
        t1.record(); torch.cuda.synchronize()
    return iters / (t0.elapsed_time(t1) / 1000), out

fps32, _ = time_torch(model, x32)
results["PyTorch fp32"] = (fps32, float("inf"))
mh = model.half().to(memory_format=torch.channels_last)
fph, outh = time_torch(mh, x32.half().to(memory_format=torch.channels_last))
results["PyTorch fp16 channels_last"] = (fph, psnr(outh.float(), ref32))

print(f"{'engine':32s} {'fps':>7s}  PSNR vs fp32 ref")
for k, (f, p) in results.items():
    print(f"{k:32s} {f:7.2f}  {p:.1f} dB")
print("ncnn (T10, earlier session, same network, 1080p x4): 1.13 fps")

In [ ]:
# 6. Look at the picture: 960x540 crop of the TensorRT output next to a bicubic upscale of the same area
y0, x0 = 1500, 3000
crop = (trt_out[0, :, y0:y0 + 540, x0:x0 + 960].clamp(0, 1) * 255).byte().permute(1, 2, 0).cpu().numpy()
src = x32[:, :, y0 // 4:y0 // 4 + 135, x0 // 4:x0 // 4 + 240]
bic = (F.interpolate(src, scale_factor=4, mode="bicubic").clamp(0, 1)[0] * 255).byte().permute(1, 2, 0).cpu().numpy()
Image.fromarray(np.concatenate([bic, crop], axis=1)).save("/kaggle/working/compare_bicubic_vs_trt.png")
Image.open("/kaggle/working/compare_bicubic_vs_trt.png")

In [ ]:
# 7. Optional: any other ONNX model (e.g. an AnimeJaNai compact model), fixed 1080p input.
# Put a path or URL here; leave empty to skip. Only static-shape or opset-compatible ONNX files will build.
extra = ""
if extra:
    path = extra
    if extra.startswith("http"):
        path = "/kaggle/temp/extra.onnx"
        sh(f"wget -q -O {path} '{extra}'")
    e2 = build_engine(path)
    f2, o2 = run_engine(e2, x32)
    print(f"extra model: {f2:.2f} fps, output {tuple(o2.shape)}")